In [1]:
# Project and symbolic tools
import heyoka as hy
import quivira as qv

# Numerics and visualization
import numpy as np
from _animation import animate_bar_model

Setup lattice

In [2]:
model = qv.bar_joint_model()

# Number of cells in each direction
n_x = 10
n_y = 10

# Cell spacing
dx = 8
dy = 8

# Original cell geometry
base_point_pairs = [
    ((0, 0), (1, 4)),
    ((1, 4), (0, 8)),
    ((0, 8), (4, 7)),
    ((4, 7), (8, 8)),
    ((8, 8), (9, 4)),
    ((9, 4), (8, 0)),
    ((8, 0), (4, -1)),
    ((4, -1), (0, 0)),
]

# Keep track of already existing bars
existing_bars = set()

top_right_bar_index = None

for i in range(n_x):
    for j in range(n_y):

        offset_x = i * dx
        offset_y = j * dy

        for k, (start, end) in enumerate(base_point_pairs):

            new_start = (start[0] + offset_x, start[1] + offset_y)
            new_end = (end[0] + offset_x, end[1] + offset_y)

            bar_key = tuple(sorted((new_start, new_end)))

            if bar_key in existing_bars:
                continue

            # Identify bar 3 in the upper-right cell
            if i == n_x - 1 and j == n_y - 1 and k == 3:
                top_right_bar_index = len(model.bars)

            model.add_bar_from_point_pair(new_start, new_end)
            existing_bars.add(bar_key)

model.infer_joints()

model.add_support(bar_index=0, s=-0.5, dof_in_x=False, dof_in_y=False)

state0 = model.get_initial_state()
model.check_constraint_fulfillment(state0)

Add torques

In [3]:
# Number of randomly applied torques
n_torques = 10

# Random number generator (seed for reproducibility)
rng = np.random.default_rng(42)

# Select random bars without repetition
torque_bar_indices = rng.choice(
    len(model.bars),
    size=n_torques,
    replace=False
)

# Random torque values: either +1 or -1
torque_values = rng.choice([-100.0, 100.0], size=n_torques)

# Add torques as Heyoka parameters
for i, bar_index in enumerate(torque_bar_indices):
    model.add_external_force(
        bar_index=int(bar_index),
        s=0.0,
        fx=0.0,
        fy=0.0,
        torque=hy.par[i]
    )

print("Torque bar indices:", torque_bar_indices)
print("Torque values:", torque_values)

Torque bar indices: [ 37 334  38 283 190 188 305  41  88 374]
Torque values: [-100.  100. -100. -100.  100.  100.  100. -100.  100.  100.]


In [4]:
ode_system, W = model.build_lagrange_eom(return_W=True)

hy.lagrangian took 37.677941 seconds
build_mass_matrix took 5.246534 seconds
build_constraint_jacobian took 2.220315 seconds
build_Jdqd took 6.841808 seconds
factorize M took 0.996140 seconds
force contribution took 0.003263 seconds
solve M @ G = J.T took 1.111708 seconds
my_matmul: A(1080, 1320) @ B(1320, 1080) -> C(1080, 1080)
build W took 0.969210 seconds
build rhs took 1.410367 seconds
gaussian_elimination took 2.872770 seconds
my_matmul: A(1320, 1080) @ B(1080, 1) -> C(1320, 1)
acceleration correction took 0.386655 seconds
find_constrained_accelerations took 7.768570 seconds


In [5]:
ta = hy.taylor_adaptive(ode_system, pars=torque_values.tolist(), compact_mode=True)

In [6]:
ta.time = 0.0
ta.state[:] = state0

t_end = 20
tgrid = np.linspace(0.0, t_end, 101)
solution = ta.propagate_grid(tgrid)[-1]

Animate

In [7]:
animate_bar_model(model, solution, pars=ta.pars, save_animation=False)

In [8]:
# Save state1 and the original applied torques
state1 = np.asarray(solution[-1], dtype=float).copy()
torques_original = np.asarray(torque_values, dtype=float).copy()

nq = len(model.q)
t1 = float(tgrid[-1])
duration = float(tgrid[-1] - tgrid[0])

# Reset integrator to state1
ta.time = t1
ta.state[:] = state1
ta.pars[:] = torques_original

# Reverse all generalized velocities, keeping positions unchanged
ta.state[nq:2*nq] *= -1.0

# Integrate forward for the same duration
tgrid2 = np.linspace(t1, t1 + duration, len(tgrid))
solution2 = ta.propagate_grid(tgrid2)[-1]
state2 = solution2[-1].copy()

# Compare state2 with state0
position_error = np.max(np.abs(state2[:nq] - state0[:nq]))
velocity_error = np.max(np.abs(state2[nq:] - state0[nq:]))

print("Maximum position error:", position_error)
print("Maximum velocity error:", velocity_error)
print("States match:", np.allclose(state2, state0, atol=1e-7, rtol=1e-7))

# Animate the full trajectory: state0 -> state1 -> state2
#full_solution = np.vstack((solution, solution2[1:]))

Maximum position error: 1.5772821949453402
Maximum velocity error: 1.4097380390594791
States match: False


In [11]:
animate_bar_model(
    model,
    solution2,
    pars=torques_original,
    save_animation=False
)

In [12]:
# Reset integrator to state1
ta.time = t1
ta.state[:] = state1

# Set all generalized velocities to zero
ta.state[nq:2*nq] = 0.0

# Reverse the applied torques
ta.pars[:] = -torques_original

# Integrate forward for the same duration
tgrid3 = np.linspace(t1, t1 + duration, len(tgrid))
solution3 = ta.propagate_grid(tgrid3)[-1]
state3 = solution3[-1].copy()

# Compare only generalized positions of state0 and state3
position_error = np.max(np.abs(state3[:nq] - state0[:nq]))
position_error_norm = np.linalg.norm(state3[:nq] - state0[:nq])

print("Maximum position error:", position_error)
print("Position error norm:", position_error_norm)
print(
    "Positions match:",
    np.allclose(state3[:nq], state0[:nq], atol=1e-7, rtol=1e-7)
)

Maximum position error: 64.09716719867673
Position error norm: 840.9408382519325
Positions match: False


In [13]:
animate_bar_model(
    model,
    solution3,
    pars=torques_original,
    save_animation=False
)